# F1 2026 Belgian GP Qualifying: Econometric Analysis of Energy Deployment

Econometric version of the battery-deployment project. Same question: does
energy deployment differ between teammates and between teams, and does it track
who's faster? But the physics constants and rule-based classification are
replaced with estimated models, so every number carries a standard error or CI.

What changed from the physics notebook:

- CdA / Crr: regressed from each car's coast phases instead of hard-coded.
- Deploy/harvest labels: Markov-switching regression instead of a throttle
  threshold, so each point gets a regime probability.
- Theoretical best lap: stochastic frontier over the whole field instead of
  `min()` of two teammates.
- Driver vs team: two-way fixed effects.
- Energy reallocation: constrained optimisation returning a shadow price
  (s per MJ) instead of a greedy loop.

Public telemetry has no battery SoC or MGU-K power, so deployment is still
inferred, not measured. Results are comparative and directional, with error bars.

## 1. Setup

`statsmodels` and `scipy` do the estimation; `fastf1` only loads data (§3-4).

In [ ]:
# %pip install fastf1 pandas numpy matplotlib statsmodels scipy

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from scipy.optimize import minimize
from scipy.stats import norm

warnings.filterwarnings("ignore")
%matplotlib inline

KMH_TO_MS = 1000.0 / 3600.0

## 2. Config

Power-unit ceilings are regulatory and fixed. `car_mass_kg` is fixed. CdA and
Crr are priors only, fallbacks used when a lap has too few coast points to
estimate from (§5).

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class PriorParams:
    # power-unit ceilings (regulatory, fixed)
    ice_power_kw: float = 400.0
    mguk_power_kw: float = 350.0
    battery_capacity_mj: float = 4.0
    max_deploy_per_lap_mj: float = 8.5
    max_harvest_rate_kw: float = 350.0

    # vehicle constants; cda/crr are priors, estimated per-car in section 5
    car_mass_kg: float = 768.0
    prior_cda: float = 0.90
    prior_crr: float = 0.012
    air_density: float = 1.225
    g: float = 9.81

PRIORS = PriorParams()

@dataclass(frozen=True)
class SessionConfig:
    year: int = 2026
    gp: str = "Belgium"
    session: str = "Q"
    cache_dir: str = "cache"

SESSION = SessionConfig()
MINI_SECTOR_LENGTH_M = 50.0

PRIORS, SESSION

## 3. Data loading

Pull the whole grid, not just teammate pairs: the frontier (§7) needs the full
field per sector and the fixed-effects model (§8) needs every driver-team combo.
Teammate pairs are still recorded for the within-car delta plots.

In [ ]:
import fastf1

def enable_cache(cache_dir=SESSION.cache_dir):
    os.makedirs(cache_dir, exist_ok=True)
    fastf1.Cache.enable_cache(cache_dir)

def load_session(year=SESSION.year, gp=SESSION.gp, session_type=SESSION.session):
    session = fastf1.get_session(year, gp, session_type)
    session.load(telemetry=True, weather=True, messages=True)
    return session

def get_all_drivers(session):
    """Return {abbr: team} for every driver in the session results."""
    res = session.results
    return dict(zip(res["Abbreviation"], res["TeamName"]))

def get_teammate_pairs(session):
    res = session.results
    return {team: g["Abbreviation"].tolist()
            for team, g in res.groupby("TeamName")}

def best_lap_telemetry(session, driver):
    """Fastest accurate qualifying lap for a driver + its aligned telemetry."""
    laps = (session.laps.pick_drivers(driver)
            if hasattr(session.laps, "pick_drivers")
            else session.laps.pick_driver(driver))
    if hasattr(laps, "pick_accurate"):
        laps = laps.pick_accurate()
    if laps.empty:
        return None
    lap = laps.pick_fastest()
    if lap is None:
        return None
    tel = lap.get_telemetry().add_distance()
    return {"driver": driver, "lap_time_s": lap["LapTime"].total_seconds(),
            "telemetry": tel}

## 4. Telemetry alignment

Interpolate each driver onto a shared distance grid so laps line up point for
point, and carry a cumulative-time channel for the downstream estimators.

In [ ]:
def _cumulative_time_s(tel):
    t = tel["Time"]
    return (t - t.iloc[0]).dt.total_seconds().to_numpy()

def resample_driver(tel, grid, suffix):
    """Interpolate one driver's channels onto the shared grid, tagged w/ suffix."""
    d = tel["Distance"].to_numpy()
    # distance can dip slightly at low speed, drop those points so interp doesn't choke
    keep = np.concatenate(([True], np.diff(d) > 0))
    d = d[keep]
    tel = tel.iloc[keep].reset_index(drop=True)
    out = {f"time_s_{suffix}": np.interp(grid, d, _cumulative_time_s(tel))}
    for ch in ["Speed", "Throttle", "Brake", "RPM", "nGear", "Z"]:
        if ch in tel.columns:
            out[f"{ch}_{suffix}"] = np.interp(grid, d, tel[ch].to_numpy().astype(float))
    return out

def build_common_grid(tel_dict, step_m=5.0):
    """Given {driver: telemetry}, return a common distance grid + per-driver frames."""
    dist_max = min(t["Distance"].max() for t in tel_dict.values())
    grid = np.arange(0, dist_max, step_m)
    frames = {}
    for drv, tel in tel_dict.items():
        cols = resample_driver(tel, grid, "X")
        df = pd.DataFrame({"Distance": grid})
        for k, v in cols.items():
            df[k.replace("_X", "")] = v
        frames[drv] = df
    return grid, frames

def align_pair(tel_a, tel_b, step_m=5.0):
    """Two-driver aligned frame with a delta-time trace (for teammate plots)."""
    dist_max = min(tel_a["Distance"].max(), tel_b["Distance"].max())
    grid = np.arange(0, dist_max, step_m)
    df = pd.DataFrame({"Distance": grid})
    for tel, sfx in [(tel_a, "A"), (tel_b, "B")]:
        for k, v in resample_driver(tel, grid, sfx).items():
            df[k] = v
    df["delta_time_s"] = df["time_s_A"] - df["time_s_B"]
    return df

## 5. Drag & rolling resistance

On a coast (throttle off, brake off) the only decelerating forces are drag (∝ v²)
and rolling resistance (constant), so per unit mass:

$$ -a_{\text{coast}} = \underbrace{\tfrac{0.5\,\rho\,C_dA}{m}}_{\beta_1} v^2 + \underbrace{C_{rr}\,g}_{\beta_0} $$

OLS of deceleration on v² over coast points: slope gives CdA, intercept gives
Crr, both with robust (HC1) SEs. Falls back to priors if too few coast points.

In [ ]:
def _coast_mask(v, thr, brk, a, throttle_max=10.0, v_min=5.0):
    """Points where the car is genuinely coasting: off throttle, off brake, slowing down."""
    brk = np.asarray(brk, dtype=float)
    # brake channel is sometimes 0/1, sometimes 0-100, normalise so "off" means the same thing
    brk_max = np.nanmax(brk) if brk.size else 1.0
    brk_norm = brk / brk_max if brk_max > 1.5 else brk
    return (thr < throttle_max) & (brk_norm < 0.02) & (a < 0) & (v > v_min)

def estimate_resistance(grid_df, suffix, mass_kg=PRIORS.car_mass_kg,
                        air_density=PRIORS.air_density, g=PRIORS.g,
                        throttle_max=10.0, min_points=30):
    """Coast-phase OLS for CdA, Crr with robust SEs; None if too few coast points.
    Adds a grade term when elevation (Z) telemetry is available, since Spa's got
    enough elevation change to otherwise bleed into the Crr estimate."""
    v = grid_df[f"Speed_{suffix}"].to_numpy() * KMH_TO_MS
    thr = grid_df[f"Throttle_{suffix}"].to_numpy()
    brk = grid_df[f"Brake_{suffix}"].to_numpy()
    t = grid_df[f"time_s_{suffix}"].to_numpy()
    a = np.gradient(v, t)

    coast = _coast_mask(v, thr, brk, a, throttle_max=throttle_max)
    if coast.sum() < min_points:
        return None

    y = -a[coast]
    v2 = v[coast] ** 2

    grade_col = f"Z_{suffix}"
    has_grade = grade_col in grid_df.columns
    if has_grade:
        dist = grid_df["Distance"].to_numpy()
        z = grid_df[grade_col].to_numpy()
        grade = np.gradient(z, dist)
        X = sm.add_constant(np.column_stack([v2, grade[coast]]))
    else:
        X = sm.add_constant(v2)

    m = sm.OLS(y, X).fit(cov_type="HC1")
    b0, b1 = m.params[0], m.params[1]
    se0, se1 = m.bse[0], m.bse[1]
    return {
        "cda": 2 * b1 * mass_kg / air_density,
        "cda_se": 2 * se1 * mass_kg / air_density,
        "crr": b0 / g, "crr_se": se0 / g,
        "n_coast": int(coast.sum()), "r2": m.rsquared, "model": m,
        "has_grade_term": has_grade,
    }

def resistance_or_prior(grid_df, suffix):
    """Estimate if possible, else fall back to priors (flagged)."""
    est = estimate_resistance(grid_df, suffix)
    if est is None:
        return {"cda": PRIORS.prior_cda, "crr": PRIORS.prior_crr,
                "cda_se": np.nan, "crr_se": np.nan, "source": "prior"}
    est["source"] = "estimated"
    return est

## 6. Wheel power + regime detection

Wheel power from `F = ma + drag + roll`, `P = F·v`, using each car's estimated
resistance from §5.

Regimes (deploy / neutral / harvest) are treated as a latent state and fit with
a Markov-switching regression on acceleration, giving a per-point probability of
each regime rather than a hard throttle cutoff. Energy is then integrated
probability-weighted.

In [ ]:
def total_wheel_power_kw(grid_df, suffix, resistance, mass_kg=PRIORS.car_mass_kg,
                         air_density=PRIORS.air_density, g=PRIORS.g):
    """Total propulsive power at the wheels (ICE + MGU-K combined)."""
    v = grid_df[f"Speed_{suffix}"].to_numpy() * KMH_TO_MS
    t = grid_df[f"time_s_{suffix}"].to_numpy()
    a = np.gradient(v, t)
    f_drag = 0.5 * air_density * resistance["cda"] * v ** 2
    f_roll = resistance["crr"] * mass_kg * g
    f_total = mass_kg * a + f_drag + f_roll
    return (f_total * v) / 1000.0

wheel_power_kw = total_wheel_power_kw

def mguk_power_kw(total_power_kw, ice_power_kw=PRIORS.ice_power_kw,
                  mguk_cap_kw=PRIORS.mguk_power_kw):
    """Electrical-only slice of total power: whatever's above what the ICE can
    make alone, capped at the MGU-K's own limit. This is what should be
    compared against the 8.5 MJ/lap cap, not the combined total."""
    return np.clip(np.asarray(total_power_kw) - ice_power_kw, 0.0, mguk_cap_kw)

def estimate_regimes(grid_df, suffix, k_regimes=3, maxiter=200):
    """Markov-switching regression on acceleration; returns per-point regime probs."""
    v = grid_df[f"Speed_{suffix}"].to_numpy() * KMH_TO_MS
    t = grid_df[f"time_s_{suffix}"].to_numpy()
    a = np.gradient(v, t)

    mod = MarkovRegression(pd.Series(a), k_regimes=k_regimes, trend="c",
                           switching_variance=True)
    res = mod.fit(maxiter=maxiter, disp=False, em_iter=20, search_reps=10)

    means = res.params[[f"const[{i}]" for i in range(k_regimes)]].to_numpy()
    order = np.argsort(means)
    if k_regimes == 3:
        names = {order[0]: "harvest", order[1]: "neutral", order[2]: "deploy"}
    else:
        names = {order[0]: "harvest", order[-1]: "deploy"}
        for m_, idx in enumerate(order[1:-1], 1):
            names[idx] = f"mid{m_}"

    prob = res.smoothed_marginal_probabilities.to_numpy()
    hard = np.array([names[i] for i in prob.argmax(axis=1)], dtype=object)
    prob_df = pd.DataFrame({names[i]: prob[:, i] for i in range(k_regimes)})
    prob_df["accel_ms2"] = a
    prob_df["label"] = hard
    return {"probabilities": prob_df, "labels": hard,
            "regime_means": {names[i]: float(means[i]) for i in range(k_regimes)},
            "aic": res.aic, "bic": res.bic, "result": res}

def energy_summary(grid_df, suffix, power_kw, regimes):
    """Integrate power over time within deploy / harvest regimes -> MJ (+ probs)."""
    t = grid_df[f"time_s_{suffix}"].to_numpy()
    dt = np.diff(t, prepend=t[0])
    p_deploy = regimes["probabilities"].get("deploy", pd.Series(np.zeros(len(t)))).to_numpy()
    p_harvest = regimes["probabilities"].get("harvest", pd.Series(np.zeros(len(t)))).to_numpy()
    electrical_kw = mguk_power_kw(power_kw)
    deployed = np.sum(electrical_kw * p_deploy * dt) / 1000.0
    harvested = np.sum(np.clip(-power_kw, 0, None) * p_harvest * dt) / 1000.0
    return {"deployed_mj": deployed, "harvested_mj": harvested,
            "net_mj": deployed - harvested}

## 7. Theoretical best lap, stochastic frontier

Pool the whole field per mini-sector and model each time as

$$ \text{time}_i = f + u_i + v_i,\qquad u_i \ge 0,\; v_i \sim N(0,\sigma_v^2) $$

with $u_i$ half-normal (can only be slower than the frontier $f$) and $v_i$
symmetric noise. Half-normal MLE per sector recovers $f$, $\sigma_u$, $\sigma_v$,
and a per-driver inefficiency estimate. Composite best = sum of sector frontiers.

In [ ]:
def _sfa_nll(params, y):
    b0, log_su, log_sv = params
    su, sv = np.exp(log_su), np.exp(log_sv)
    eps = y - b0
    sigma = np.sqrt(su**2 + sv**2)
    lam = su / sv
    ll = (np.log(2) - np.log(sigma) + norm.logpdf(eps / sigma)
          + norm.logcdf(eps * lam / sigma))
    return -np.sum(ll)

def fit_sector_frontier(times, undercut_tol=0.98, n_restarts=4, seed=0):
    """Half-normal cost-frontier MLE for one sector's field of times.
    Tries a few starting points and throws out fits that don't converge or
    that undercut the sector's fastest actual time by more than 2%, falling
    back to the observed minimum instead."""
    y = np.asarray(times, float); y = y[np.isfinite(y)]
    if len(y) < 4:
        return None

    rng = np.random.default_rng(seed)
    best_fit = None
    for k in range(n_restarts):
        jitter = 1.0 if k == 0 else rng.uniform(0.5, 1.5)
        x0 = [y.min(), np.log(y.std() * jitter + 1e-6), np.log(y.std() * jitter + 1e-6)]
        r = minimize(_sfa_nll, x0, args=(y,), method="Nelder-Mead",
                     options={"xatol": 1e-6, "fatol": 1e-6, "maxiter": 2000})
        if not r.success:
            continue
        if best_fit is None or r.fun < best_fit.fun:
            best_fit = r

    if best_fit is None or best_fit.x[0] < undercut_tol * y.min():
        return {"frontier": float(y.min()), "sigma_u": np.nan, "sigma_v": np.nan,
                "u_hat": None, "converged": False, "loglik": np.nan,
                "fallback": True}

    b0, log_su, log_sv = best_fit.x
    su, sv = np.exp(log_su), np.exp(log_sv)
    s2 = su**2 + sv**2
    eps = y - b0
    sig_star = su * sv / np.sqrt(s2)
    lam = (eps * su**2 / s2) / sig_star
    u_hat = sig_star * (norm.pdf(lam) / (norm.cdf(lam) + 1e-12) + lam)
    return {"frontier": b0, "sigma_u": su, "sigma_v": sv,
            "u_hat": u_hat, "converged": True, "loglik": -best_fit.fun,
            "fallback": False}

def build_sector_panel(grid, frames, sector_length_m=MINI_SECTOR_LENGTH_M):
    """Long panel: one row per driver per mini-sector with that sector's time."""
    rows = []
    for drv, df in frames.items():
        sec = (df["Distance"] // sector_length_m).astype(int)
        tcol = "time_s"
        for sid, g in df.assign(sector_id=sec).groupby("sector_id"):
            rows.append({"driver": drv, "sector_id": int(sid),
                         "sector_time_s": g[tcol].iloc[-1] - g[tcol].iloc[0],
                         "avg_speed_kmh": g["Speed"].mean()})
    return pd.DataFrame(rows)

def frontier_best_lap(sector_panel):
    """Fit a frontier per sector; sum frontiers -> composite theoretical best."""
    out, total, n_fallback = [], 0.0, 0
    n_sectors = sector_panel["sector_id"].nunique()
    for sid, g in sector_panel.groupby("sector_id"):
        fr = fit_sector_frontier(g["sector_time_s"].to_numpy())
        if fr is None:
            best = g["sector_time_s"].min()
            out.append({"sector_id": sid, "frontier_s": best,
                        "sigma_u": np.nan, "sigma_v": np.nan})
            total += best
            n_fallback += 1
            continue
        if fr.get("fallback"):
            n_fallback += 1
        out.append({"sector_id": sid, "frontier_s": fr["frontier"],
                    "sigma_u": fr["sigma_u"], "sigma_v": fr["sigma_v"]})
        total += fr["frontier"]
    print(f"[frontier] fit {n_sectors - n_fallback}/{n_sectors} sectors directly; "
          f"{n_fallback} fell back to the observed minimum.")
    return pd.DataFrame(out), total

## 8. Driver vs team, two-way fixed effects

Regress mini-sector time on sector + team + driver fixed effects:

$$ \text{time}_{d,s} = \alpha_s + \tau_{\text{team}(d)} + \delta_d + \varepsilon_{d,s} $$

$\delta_d$ is driver pace net of track layout and car.

Within one session team and driver FE are collinear (nobody changes car
mid-weekend), so they can't be separated, statsmodels drops redundant levels.
Separating car from driver needs movers (drivers who changed team), i.e. pooling
several seasons. Runs on one session, but read the single-session driver effects
as car-confounded.

In [ ]:
import patsy

def fit_two_way_fe(panel, value="sector_time_s"):
    """time ~ C(sector) + C(team) + C(driver), cluster-robust by sector.
    One session means team is fully determined by driver (nobody swaps cars),
    so the three-way formula is rank-deficient and statsmodels won't tell you,
    it just silently returns garbage. Checks the rank first and, if it's short,
    fits sector+driver and sector+team separately instead."""
    df = panel.dropna(subset=[value]).copy()
    if "team" not in df.columns:
        raise ValueError("panel needs a 'team' column (map driver->team).")

    def effects(res, prefix):
        rows = [{"level": n.split("T.")[-1].rstrip("]"), "effect": c,
                 "se": res.bse[n], "t": res.tvalues[n], "p": res.pvalues[n]}
                for n, c in res.params.items() if n.startswith(prefix)]
        return pd.DataFrame(rows).sort_values("effect").reset_index(drop=True)

    full_formula = f"{value} ~ C(sector_id) + C(team) + C(driver)"
    _, X_full = patsy.dmatrices(full_formula, data=df, return_type="dataframe")
    rank = np.linalg.matrix_rank(X_full.to_numpy())
    identified = rank == X_full.shape[1]

    if identified:
        res = smf.ols(full_formula, data=df).fit(
            cov_type="cluster", cov_kwds={"groups": df["driver"]})
        return {"result": res, "driver_effects": effects(res, "C(driver)"),
                "team_effects": effects(res, "C(team)"), "n_obs": len(df),
                "identified": True}

    print(f"[two-way FE] rank-deficient ({rank}/{X_full.shape[1]}), "
          f"fitting sector+driver and sector+team separately.")
    res_driver = smf.ols(f"{value} ~ C(sector_id) + C(driver)", data=df).fit(
        cov_type="cluster", cov_kwds={"groups": df["sector_id"]})
    res_team = smf.ols(f"{value} ~ C(sector_id) + C(team)", data=df).fit(
        cov_type="cluster", cov_kwds={"groups": df["sector_id"]})
    return {"result": res_driver, "team_result": res_team,
            "driver_effects": effects(res_driver, "C(driver)"),
            "team_effects": effects(res_team, "C(team)"),
            "n_obs": len(df), "identified": False}

def add_team_column(panel, driver_team_map):
    p = panel.copy()
    p["team"] = p["driver"].map(driver_team_map)
    return p

## 9. Energy reallocation, constrained optimisation

Maximise total time saved subject to a total-energy budget $B$:

$$ \max_{\{e_i\}} \sum_i \frac{L_i e_i}{m v_i^3} \quad\text{s.t.}\quad \sum_i e_i \le B,\; 0 \le e_i \le \bar e_i $$

At the optimum every zone getting energy has equal marginal return, that common
value is the Lagrange multiplier, the shadow price of energy (s per MJ). Solved
by water-filling.

Budget defaults to the driver's own estimated deployed energy (answering "did you
place your energy well"). The physics version defaulted to
`min(cap, harvested + capacity)`, which on the real run was ~2x actual deployment
and flagged every driver implausible. Pass `budget_mj` to override.

In [ ]:
def identify_zones(grid_df, suffix, power_kw, regimes):
    """Contiguous non-harvest zones where extra deployment could help."""
    labels = regimes["labels"]
    can_deploy = labels != "harvest"
    dist = grid_df["Distance"].to_numpy()
    t = grid_df[f"time_s_{suffix}"].to_numpy()
    electrical_kw = mguk_power_kw(power_kw)
    zones, i, n = [], 0, len(can_deploy)
    while i < n:
        if can_deploy[i]:
            j = i
            while j < n and can_deploy[j]:
                j += 1
            end = min(j, n - 1)
            dur = max(t[end] - t[i], 1e-6)
            length = max(dist[end] - dist[i], 1e-6)
            base = float(np.mean(electrical_kw[i:end])) if end > i else 0.0
            head = max(PRIORS.mguk_power_kw - base, 0.0)
            zones.append({"length_m": length, "avg_speed_ms": length / dur,
                          "max_extra_mj": head * 1000.0 * dur / 1e6,
                          "distance_start_m": float(dist[i])})
            i = j
        else:
            i += 1
    return zones

def _zone_time_saved(alloc_mj, L, v, mass_kg):
    """Time saved from adding energy to a zone, raising its speed from v to
    sqrt(v^2 + 2E/m) instead of assuming a constant s-per-MJ rate throughout,
    which only holds for small E and blows up otherwise."""
    e_j = np.asarray(alloc_mj, dtype=float) * 1e6
    v_new = np.sqrt(np.maximum(v**2 + 2 * e_j / mass_kg, 1e-6))
    return L / v - L / v_new

def optimize_with_shadow_price(zones, budget_mj, mass_kg=PRIORS.car_mass_kg):
    """Water-filling allocation; returns allocation, time saved, shadow price."""
    if not zones or budget_mj <= 0:
        return {"allocation_mj": [0.0] * len(zones), "time_saved_s": 0.0,
                "shadow_price_s_per_mj": 0.0, "budget_used_mj": 0.0}
    L = np.array([z["length_m"] for z in zones])
    v = np.array([max(z["avg_speed_ms"], 1.0) for z in zones])
    cap = np.array([z["max_extra_mj"] for z in zones])
    g = L * 1e6 / (mass_kg * v ** 3)          # marginal s per MJ at E=0, ranks zones only
    order = np.argsort(-g)
    alloc = np.zeros(len(zones)); remaining = budget_mj; shadow = 0.0
    for idx in order:
        if remaining <= 1e-9:
            break
        take = min(cap[idx], remaining)
        alloc[idx] = take; remaining -= take
        if take > 0:
            v_after = np.sqrt(max(v[idx] ** 2 + 2 * take * 1e6 / mass_kg, 1e-6))
            shadow = L[idx] * 1e6 / (mass_kg * v_after ** 3)
    saved = float(np.sum(_zone_time_saved(alloc, L, v, mass_kg)))
    return {"allocation_mj": alloc.tolist(), "time_saved_s": saved,
            "shadow_price_s_per_mj": float(shadow),
            "budget_used_mj": float(budget_mj - remaining),
            "zones": zones}

def optimal_lap_report(driver, actual_lap_s, grid_df, suffix, power_kw, regimes,
                       energy, budget_mj=None):
    if budget_mj is None:
        budget_mj = min(energy["deployed_mj"], PRIORS.max_deploy_per_lap_mj)
    zones = identify_zones(grid_df, suffix, power_kw, regimes)
    opt = optimize_with_shadow_price(zones, budget_mj)
    saved = opt["time_saved_s"]
    pct = saved / actual_lap_s if actual_lap_s else 0
    return {"driver": driver, "actual_lap_s": actual_lap_s,
            "estimated_optimal_lap_s": actual_lap_s - saved,
            "time_saved_s": saved, "pct_of_lap": pct,
            "plausible": pct < 0.02,
            "shadow_price_s_per_mj": opt["shadow_price_s_per_mj"],
            "budget_mj": budget_mj}

## 10. Plots

Resistance fits show their scatter, regimes are shaded by probability, driver
effects come with 95% CIs.

In [ ]:
PLOTS_DIR = "Spa 2026 figures"
os.makedirs(PLOTS_DIR, exist_ok=True)

def _save(fig, path):
    if path:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        fig.savefig(path, dpi=150, bbox_inches="tight")

def plot_resistance_fit(grid_df, suffix, resistance, driver="", save_path=None):
    v = grid_df[f"Speed_{suffix}"].to_numpy() * KMH_TO_MS
    thr = grid_df[f"Throttle_{suffix}"].to_numpy()
    brk = grid_df[f"Brake_{suffix}"].to_numpy()
    t = grid_df[f"time_s_{suffix}"].to_numpy()
    a = np.gradient(v, t)
    coast = _coast_mask(v, thr, brk, a)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter((v[coast])**2, -a[coast], s=6, alpha=0.4, label="coast points")
    xs = np.linspace(0, (v[coast]**2).max(), 100) if coast.any() else np.linspace(0, 1, 2)
    b1 = resistance["cda"] * PRIORS.air_density / (2 * PRIORS.car_mass_kg)
    b0 = resistance["crr"] * PRIORS.g
    ax.plot(xs, b0 + b1 * xs, "r-", lw=2, label="fitted drag law")
    ax.set_xlabel("v^2 (m^2/s^2)"); ax.set_ylabel("deceleration (m/s^2)")
    ax.set_title(f"{driver}: coast-phase resistance fit "
                 f"(CdA={resistance['cda']:.2f}, Crr={resistance['crr']:.4f})")
    ax.legend(); fig.tight_layout(); _save(fig, save_path); return fig

def plot_regime_probs(grid_df, regimes, driver="", save_path=None):
    prob = regimes["probabilities"]
    fig, ax = plt.subplots(figsize=(12, 3.2))
    dist = grid_df["Distance"].to_numpy()
    colors = {"deploy": "#2ca02c", "harvest": "#1f77b4", "neutral": "#d3d3d3"}
    bottom = np.zeros(len(dist))
    for name in ["harvest", "neutral", "deploy"]:
        if name in prob:
            ax.fill_between(dist, bottom, bottom + prob[name].to_numpy(),
                            color=colors.get(name, "grey"), alpha=0.8, label=name)
            bottom = bottom + prob[name].to_numpy()
    ax.set_ylim(0, 1); ax.set_xlabel("Distance (m)")
    ax.set_ylabel("P(regime)"); ax.set_title(f"{driver}: regime probabilities")
    ax.legend(loc="upper right", ncol=3, fontsize=8)
    fig.tight_layout(); _save(fig, save_path); return fig

def plot_driver_effects(fe, save_path=None):
    de = fe["driver_effects"]
    fig, ax = plt.subplots(figsize=(8, max(3, 0.35 * len(de))))
    ax.errorbar(de["effect"], range(len(de)), xerr=1.96 * de["se"],
                fmt="o", capsize=3)
    ax.set_yticks(range(len(de))); ax.set_yticklabels(de["level"])
    ax.axvline(0, color="grey", ls="--", lw=0.8)
    ax.set_xlabel("Driver effect on sector time (s)  [lower = faster]")
    title = "Driver fixed effects (95% CI)"
    if not fe.get("identified", True):
        title += "\n(car-confounded: team dropped, single session)"
    ax.set_title(title)
    fig.tight_layout(); _save(fig, save_path); return fig

def plot_delta_time(aligned, a, b, save_path=None):
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(aligned["Distance"], aligned["delta_time_s"], "k", lw=1.3)
    ax.axhline(0, color="grey", ls="--", lw=0.8)
    ax.fill_between(aligned["Distance"], aligned["delta_time_s"], 0,
                    where=aligned["delta_time_s"] > 0, color="tab:red", alpha=0.25,
                    label=f"{a} losing")
    ax.fill_between(aligned["Distance"], aligned["delta_time_s"], 0,
                    where=aligned["delta_time_s"] < 0, color="tab:green", alpha=0.25,
                    label=f"{a} gaining")
    ax.set_xlabel("Distance (m)"); ax.set_ylabel(f"delta s [{a}-{b}]")
    ax.set_title(f"Lap delta: {a} vs {b}"); ax.legend(loc="upper left")
    fig.tight_layout(); _save(fig, save_path); return fig

## 11. Self-test

Checks the pipeline runs before pulling real data. The synthetic lap is built
from a geometric speed profile, not real physics, so every magnitude it prints
(resistance, energy, time saved, shadow price) is meaningless by construction,
this only confirms nothing crashes. Real numbers come from §12.

In [ ]:
def _make_synthetic_lap(seed, corner_speeds_kmh, straight_boost=1.0,
                        n_points=2000, lap_length_m=7004.0):
    rng = np.random.default_rng(seed)
    n_corners = len(corner_speeds_kmh)
    seg_len = lap_length_m / n_corners
    distances = np.linspace(0, lap_length_m, n_points)
    speed = np.zeros(n_points)
    for i, d in enumerate(distances):
        seg = int(d // seg_len) % n_corners
        pos = (d % seg_len) / seg_len
        va, vn = corner_speeds_kmh[seg], corner_speeds_kmh[(seg + 1) % n_corners]
        vp = min(340.0, va + straight_boost * (200 + rng.normal(0, 10)))
        v = va + (pos / 0.55) * (vp - va) if pos < 0.55 else vp + ((pos - 0.55) / 0.45) * (vn - vp)
        speed[i] = max(v, 60.0)
    speed = np.clip(speed + rng.normal(0, 1.5, n_points), 50, 340)
    dv = np.gradient(speed)
    thr = np.clip((dv > 0) * 100 + rng.normal(0, 3, n_points), 0, 100)
    brk = np.clip((dv < -3) * 1.0, 0, 1)
    v_ms = speed * KMH_TO_MS
    dt = np.diff(distances) / np.clip((v_ms[:-1] + v_ms[1:]) / 2, 5, None)
    t = np.concatenate([[0], np.cumsum(dt)])
    df = pd.DataFrame({"Distance": distances, "time_s": t, "Speed": speed,
                       "Throttle": thr, "Brake": brk})
    return df, t[-1]

# synthetic field: 6 cars, 3 teams
corner_speeds = [80, 120, 200, 70, 150, 260, 90, 180, 60, 220]
synth = {}
team_of = {}
teams = {"Alpha": (0, 1.02), "Bravo": (2, 1.00), "Charlie": (4, 0.98)}
drv_i = 0
for team, (seed0, boost) in teams.items():
    for k in range(2):
        name = f"{team[0]}{k+1}"
        df, lap = _make_synthetic_lap(seed0 + k, corner_speeds,
                                      straight_boost=boost * (1 + 0.01 * k))
        synth[name] = {"telemetry_like": df, "lap_time_s": lap}
        team_of[name] = team
        drv_i += 1

# put each on a common grid
grid = np.arange(0, min(d["telemetry_like"]["Distance"].max() for d in synth.values()), 5.0)
frames = {}
for name, d in synth.items():
    src = d["telemetry_like"]
    df = pd.DataFrame({"Distance": grid})
    df["time_s"] = np.interp(grid, src["Distance"], src["time_s"])
    df["time_s_X"] = df["time_s"]
    for ch in ["Speed", "Throttle", "Brake"]:
        df[ch + "_X"] = np.interp(grid, src["Distance"], src[ch])
        df[ch] = df[ch + "_X"]
    frames[name] = df

print("SELF-TEST: econometric pipeline")
print("-" * 60)

# [1] resistance on one car
one = frames["A1"]
res = estimate_resistance(one, "X")
print(f"[1] resistance A1: CdA={res['cda']:.2f}+/-{res['cda_se']:.2f}, "
      f"Crr={res['crr']:.3f}, R2={res['r2']:.2f}, n_coast={res['n_coast']}")

# [2] regimes on one car
reg = estimate_regimes(one, "X", k_regimes=3)
u, c = np.unique(reg["labels"], return_counts=True)
print(f"[2] regimes A1: {dict(zip(u, c.tolist()))}, AIC={reg['aic']:.0f}")

# [3] power + energy
pw = wheel_power_kw(one, "X", res)
en = energy_summary(one, "X", pw, reg)
print(f"[3] energy A1: deployed={en['deployed_mj']:.2f}MJ, "
      f"harvested={en['harvested_mj']:.2f}MJ")

# [4] stochastic frontier across the field
panel = build_sector_panel(grid, frames)
fr_df, best = frontier_best_lap(panel)
print(f"[4] frontier best lap: {best:.2f}s vs field min "
      f"{min(d['lap_time_s'] for d in synth.values()):.2f}s "
      f"(sectors fit: {fr_df['sigma_u'].notna().sum()}/{len(fr_df)})")

# [5] two-way FE with the synthetic teams (single 'season' -> car-confounded)
panel_fe = add_team_column(panel, team_of)
fe = fit_two_way_fe(panel_fe)
print(f"[5] two-way FE: n_obs={fe['n_obs']}, "
      f"driver effects={len(fe['driver_effects'])} "
      f"(single session -> team/driver partly collinear, as documented)")

# [6] constrained optimization + shadow price
opt = optimal_lap_report("A1", synth["A1"]["lap_time_s"], one, "X", pw, reg, en)
print(f"[6] optimal A1: saved {opt['time_saved_s']:.3f}s "
      f"({opt['pct_of_lap']*100:.2f}% of lap), "
      f"shadow price={opt['shadow_price_s_per_mj']:.3f} s/MJ, "
      f"plausible={opt['plausible']}")

assert best <= min(d["lap_time_s"] for d in synth.values()) + 5
print("-" * 60)
print("SELF-TEST PASSED: pipeline runs end-to-end.")

## 12. Real session

Loads the field, builds the pooled panel, runs every estimator, and produces
per-driver diagnostics plus teammate delta plots.

In [ ]:
enable_cache()
print(f"Loading {SESSION.year} {SESSION.gp} {SESSION.session}...")
session = load_session()
driver_team = get_all_drivers(session)
pairs = get_teammate_pairs(session)
print(f"{len(driver_team)} drivers, {len(pairs)} teams")
driver_team

In [ ]:
# fastest-lap telemetry for every driver
tel = {}
for drv in driver_team:
    r = best_lap_telemetry(session, drv)
    if r is not None:
        tel[drv] = r["telemetry"]
lap_times = {drv: best_lap_telemetry(session, drv)["lap_time_s"]
             for drv in tel}
print(f"Got telemetry for {len(tel)} drivers.")

# common grid + aligned frames
grid, frames = build_common_grid(tel, step_m=5.0)
print(f"Common grid: {len(grid)} points over {grid.max():.0f} m")

In [ ]:
# per-driver: resistance, regimes, energy
per_driver = {}
for drv, df in frames.items():
    # re-tag columns with a suffix so the estimators can find them
    dd = df.rename(columns={c: f"{c}_D" for c in
                            ["Speed", "Throttle", "Brake", "time_s"]})
    res = resistance_or_prior(dd, "D")
    reg = estimate_regimes(dd, "D", k_regimes=3)
    pw = wheel_power_kw(dd, "D", res)
    en = energy_summary(dd, "D", pw, reg)
    per_driver[drv] = {"frame": dd, "resistance": res, "regimes": reg,
                       "power_kw": pw, "energy": en}
    src = "est" if res["source"] == "estimated" else "PRIOR"
    print(f"{drv}: CdA={res['cda']:.2f}({src}), "
          f"deployed={en['deployed_mj']:.2f}MJ, harvested={en['harvested_mj']:.2f}MJ")

In [ ]:
# field: stochastic frontier best lap
panel = build_sector_panel(grid, frames)
fr_df, best_lap = frontier_best_lap(panel)
print(f"Stochastic-frontier theoretical best lap: {best_lap:.3f}s")
print(f"Fastest actual: {min(lap_times.values()):.3f}s "
      f"({min(lap_times, key=lap_times.get)})")

# field: two-way fixed effects (single session, car-confounded)
panel_fe = add_team_column(panel, driver_team)
fe = fit_two_way_fe(panel_fe)
print(f"\nTwo-way FE on {fe['n_obs']} driver-sector obs.")
print("one session: team & driver FE partly collinear; pool seasons with movers to separate")
fe["driver_effects"]

In [ ]:
# per-driver optimal lap + shadow price (own-energy budget)
opt_rows = []
for drv, d in per_driver.items():
    opt = optimal_lap_report(drv, lap_times[drv], d["frame"], "D",
                             d["power_kw"], d["regimes"], d["energy"])
    opt_rows.append({"driver": drv, "actual_s": lap_times[drv],
                     "optimal_s": opt["estimated_optimal_lap_s"],
                     "saved_s": opt["time_saved_s"],
                     "pct": opt["pct_of_lap"] * 100,
                     "shadow_s_per_mj": opt["shadow_price_s_per_mj"],
                     "plausible": opt["plausible"]})
opt_df = pd.DataFrame(opt_rows).sort_values("actual_s").reset_index(drop=True)
print("Optimal-lap reallocation (own-energy budget):")
opt_df

In [ ]:
# plots
plot_driver_effects(fe, save_path=f"{PLOTS_DIR}/driver_effects.png"); plt.show()

# per-driver diagnostics for one example driver
example = list(per_driver)[0]
d = per_driver[example]
plot_resistance_fit(d["frame"], "D", d["resistance"], driver=example,
                    save_path=f"{PLOTS_DIR}/{example}_resistance.png"); plt.show()
plot_regime_probs(d["frame"], d["regimes"], driver=example,
                  save_path=f"{PLOTS_DIR}/{example}_regimes.png"); plt.show()

# teammate delta plots
for team, drivers in pairs.items():
    if len(drivers) == 2 and all(dd in tel for dd in drivers):
        a, b = drivers
        aligned = align_pair(tel[a], tel[b])
        plot_delta_time(aligned, a, b,
                        save_path=f"{PLOTS_DIR}/{team.replace(' ', '_')}_delta.png")
        plt.show()
print(f"Plots saved under ./{PLOTS_DIR}/")

## 13. Reading the output

- Resistance (§5): teammates with overlapping CdA SEs show no measurable aero
  difference from this method.
- Regimes (§6): deploy/harvest are probabilities; the stacked area shows where
  intent is clear vs borderline.
- Frontier (§7): `sigma_u` vs `sigma_v` per sector = real pace spread vs noise.
- Driver effects (§8): CI clear of zero and of a teammate's = faster net of
  layout, but car is confounded in a single session.
- Shadow price (§9): s per extra MJ. High = energy-constrained where it mattered,
  low = more energy wouldn't have helped.

For the clean driver-vs-car split, pool 2024-2026 into one panel keeping
driver/team/season columns and re-run §8. The movers identify the separation.

## 14. Norris/Verstappen check
Their CdA still looks off. Quick check: is it just noise, or something real?

In [ ]:
def bootstrap_resistance(grid_df, suffix, mass_kg=PRIORS.car_mass_kg,
                         air_density=PRIORS.air_density, g=PRIORS.g,
                         throttle_max=10.0, n_boot=300, seed=0):
    """Resample coast points and refit CdA/Crr a few hundred times to see how
    stable the estimate actually is. Wide spread = not enough signal."""
    v = grid_df[f"Speed_{suffix}"].to_numpy() * KMH_TO_MS
    thr = grid_df[f"Throttle_{suffix}"].to_numpy()
    brk = grid_df[f"Brake_{suffix}"].to_numpy()
    t = grid_df[f"time_s_{suffix}"].to_numpy()
    a = np.gradient(v, t)
    coast = _coast_mask(v, thr, brk, a, throttle_max=throttle_max)
    n = int(coast.sum())
    if n < 10:
        return None

    y_all = -a[coast]
    v2_all = v[coast] ** 2
    grade_col = f"Z_{suffix}"
    has_grade = grade_col in grid_df.columns
    grade_all = None
    if has_grade:
        dist = grid_df["Distance"].to_numpy()
        z = grid_df[grade_col].to_numpy()
        grade_all = np.gradient(z, dist)[coast]

    rng = np.random.default_rng(seed)
    cdas, crrs = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        y = y_all[idx]; v2 = v2_all[idx]
        X = (sm.add_constant(np.column_stack([v2, grade_all[idx]]))
             if has_grade else sm.add_constant(v2))
        try:
            m = sm.OLS(y, X).fit()
            b0, b1 = m.params[0], m.params[1]
            cdas.append(2 * b1 * mass_kg / air_density)
            crrs.append(b0 / g)
        except Exception:
            continue

    cdas, crrs = np.array(cdas), np.array(crrs)
    if len(cdas) < 10:
        return None
    return {
        "n_coast": n, "n_boot_success": len(cdas),
        "cda_median": float(np.median(cdas)),
        "cda_ci90": (float(np.percentile(cdas, 5)), float(np.percentile(cdas, 95))),
        "cda_std": float(np.std(cdas)),
        "crr_median": float(np.median(crrs)),
        "crr_ci90": (float(np.percentile(crrs, 5)), float(np.percentile(crrs, 95))),
        "crr_std": float(np.std(crrs)),
    }

def coast_point_report(per_driver, throttle_max=10.0):
    """n_coast, R^2 and where each driver's coast points sit on track, with a
    z-score flagging anyone whose average location looks off vs the field."""
    rows = []
    for drv, d in per_driver.items():
        df = d["frame"]
        v = df["Speed_D"].to_numpy() * KMH_TO_MS
        thr = df["Throttle_D"].to_numpy()
        brk = df["Brake_D"].to_numpy()
        t = df["time_s_D"].to_numpy()
        a = np.gradient(v, t)
        coast = _coast_mask(v, thr, brk, a, throttle_max=throttle_max)
        dist = df["Distance"].to_numpy()[coast]
        res = d["resistance"]
        rows.append({
            "driver": drv, "n_coast": int(coast.sum()),
            "r2": res.get("r2", np.nan), "source": res["source"],
            "coast_dist_mean_m": float(dist.mean()) if len(dist) else np.nan,
            "coast_dist_std_m": float(dist.std()) if len(dist) else np.nan,
        })
    out = pd.DataFrame(rows)
    field_mean = out["coast_dist_mean_m"].mean()
    field_std = out["coast_dist_mean_m"].std()
    out["location_z"] = (out["coast_dist_mean_m"] - field_mean) / field_std
    return out.sort_values("n_coast").reset_index(drop=True)

In [ ]:
report = coast_point_report(per_driver)
print(report)

for drv in ["NOR", "VER", "PER"]:
    if drv in per_driver:
        print(f"\n{drv}:", bootstrap_resistance(per_driver[drv]["frame"], "D"))